# Marketing Analytics - Exploratory Data Analysis

This notebook provides a comprehensive exploratory data analysis of the iFood marketing campaign dataset. We'll examine customer demographics, purchasing behavior, campaign responses, and identify key patterns that could inform marketing strategy.

## Table of Contents
1. Data Loading and Initial Inspection
2. Data Quality Assessment  
3. Univariate Analysis
4. Customer Demographics
5. Purchase Behavior Analysis
6. Campaign Performance
7. Correlation Analysis
8. Bivariate Relationships
9. Outlier Detection
10. Key Findings

## 1. Data Loading and Initial Inspection

First, let's import the necessary libraries and load our dataset. We'll use Plotly for interactive visualizations throughout this analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Plotly template
px.defaults.template = "plotly_white"

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load the dataset
ifood_df = pd.read_csv("../data/raw/ifood_df.csv")

print(f"Dataset shape: {ifood_df.shape}")
print(f"Number of customers: {ifood_df.shape[0]:,}")
print(f"Number of features: {ifood_df.shape[1]}")

Dataset shape: (2205, 39)
Number of customers: 2,205
Number of features: 39


In [3]:
# First few rows
ifood_df.head()

,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,marital_Together,marital_Widow,education_2n Cycle,education_Basic,education_Graduation,education_Master,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall
0,58138.0,0,0,58,635,88,546,172,88,88,...,0,0,0,0,1,0,0,1529,1441,0
1,46344.0,1,1,38,11,1,6,2,1,6,...,0,0,0,0,1,0,0,21,15,0
2,71613.0,0,0,26,426,49,127,111,21,42,...,1,0,0,0,1,0,0,734,692,0
3,26646.0,1,0,26,11,4,20,10,3,5,...,1,0,0,0,1,0,0,48,43,0
4,58293.0,1,0,94,173,43,118,46,27,15,...,0,0,0,0,0,0,1,407,392,0


In [4]:
# Dataset information
ifood_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Income                2205 non-null   float64
 1   Kidhome               2205 non-null   int64  
 2   Teenhome              2205 non-null   int64  
 3   Recency               2205 non-null   int64  
 4   MntWines              2205 non-null   int64  
 5   MntFruits             2205 non-null   int64  
 6   MntMeatProducts       2205 non-null   int64  
 7   MntFishProducts       2205 non-null   int64  
 8   MntSweetProducts      2205 non-null   int64  
 9   MntGoldProds          2205 non-null   int64  
 10  NumDealsPurchases     2205 non-null   int64  
 11  NumWebPurchases       2205 non-null   int64  
 12  NumCatalogPurchases   2205 non-null   int64  
 13  NumStorePurchases     2205 non-null   int64  
 14  NumWebVisitsMonth     2205 non-null   int64  
 15  AcceptedCmp3         

## 2. Data Quality Assessment

Let's check for missing values, outliers, and data quality issues. **Income** is a critical feature for customer segmentation, so we'll pay special attention to it. We'll check for missing values and unusually low incomes that might indicate data quality issues.

In [5]:
# Check for missing values
missing_values = ifood_df.isnull().sum()
missing_pct = (missing_values / len(ifood_df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

print("Missing Values Summary:")
print(missing_df[missing_df['Missing_Count'] > 0])

if missing_df['Missing_Count'].sum() == 0:
    print("\n✓ No missing values detected in the dataset!")
else:
    print(f"\n⚠ Total missing values: {missing_df['Missing_Count'].sum()}")

Missing Values Summary:
Empty DataFrame
Columns: [Missing_Count, Percentage]
Index: []

✓ No missing values detected in the dataset!


In [6]:
# Special attention to Income - check for missing or anomalous values
print("Income Variable Analysis:")
print(f"Missing values: {ifood_df['Income'].isna().sum()}")
print(f"Minimum income: ${ifood_df['Income'].min():,.2f}")
print(f"Maximum income: ${ifood_df['Income'].max():,.2f}")
print(f"Mean income: ${ifood_df['Income'].mean():,.2f}")
print(f"Median income: ${ifood_df['Income'].median():,.2f}")

# Check for very low incomes that might be data quality issues
low_income_threshold = 10000
low_income_count = (ifood_df['Income'] < low_income_threshold).sum()
print(f"\nCustomers with income < ${low_income_threshold:,}: {low_income_count} ({low_income_count/len(ifood_df)*100:.2f}%)")

print("\n📊 Decision: No missing values in Income. Some very low income values exist (<$10k),")
print("   but they appear to be valid. We'll keep all data but flag outliers for modeling.")

Income Variable Analysis:
Missing values: 0
Minimum income: $1,730.00
Maximum income: $113,734.00
Mean income: $51,622.09
Median income: $51,287.00

Customers with income < $10,000: 29 (1.32%)

📊 Decision: No missing values in Income. Some very low income values exist (<$10k),
   but they appear to be valid. We'll keep all data but flag outliers for modeling.


In [7]:
# Statistical summary
ifood_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Income,2205.0,51622.094785,20713.063826,1730.0,35196.0,51287.0,68281.0,113734.0
Kidhome,2205.0,0.442177,0.537132,0.0,0.0,0.0,1.0,2.0
Teenhome,2205.0,0.506576,0.544380,0.0,0.0,0.0,1.0,2.0
Recency,2205.0,49.009070,28.932111,0.0,24.0,49.0,74.0,99.0
MntWines,2205.0,306.164626,337.493839,0.0,24.0,178.0,507.0,1493.0
MntFruits,2205.0,26.403175,39.784484,0.0,2.0,8.0,33.0,199.0
MntMeatProducts,2205.0,165.312018,217.784507,0.0,16.0,68.0,232.0,1725.0
MntFishProducts,2205.0,37.756463,54.824635,0.0,3.0,12.0,50.0,259.0
MntSweetProducts,2205.0,27.128345,41.130468,0.0,1.0,8.0,34.0,262.0
MntGoldProds,2205.0,44.057143,51.736211,0.0,9.0,25.0,56.0,321.0


## 3. Univariate Analysis

Let's examine the distribution of key numerical features using interactive Plotly visualizations. These distributions will help us understand the typical customer profile and identify any skewness or unusual patterns.

In [8]:
# Income distribution
fig = px.histogram(ifood_df, x='Income', nbins=50,
                   title='Distribution of Customer Income',
                   labels={'Income': 'Annual Income ($)', 'count': 'Number of Customers'},
                   color_discrete_sequence=['steelblue'])
fig.add_vline(x=ifood_df['Income'].median(), line_dash="dash", 
              line_color="red", annotation_text=f"Median: ${ifood_df['Income'].median():,.0f}")
fig.update_layout(showlegend=False, height=500)
fig.show()

In [9]:
# Age distribution
fig = px.histogram(ifood_df, x='Age', nbins=40,
                   title='Distribution of Customer Age',
                   labels={'Age': 'Age (years)', 'count': 'Number of Customers'},
                   color_discrete_sequence=['coral'])
fig.add_vline(x=ifood_df['Age'].median(), line_dash="dash", 
              line_color="red", annotation_text=f"Median: {ifood_df['Age'].median():.0f} years")
fig.update_layout(showlegend=False, height=500)
fig.show()

In [10]:
# Total spending distribution
fig = px.histogram(ifood_df, x='MntTotal', nbins=50,
                   title='Distribution of Total Customer Spending (Last 2 Years)',
                   labels={'MntTotal': 'Total Spending ($)', 'count': 'Number of Customers'},
                   color_discrete_sequence=['seagreen'])
fig.add_vline(x=ifood_df['MntTotal'].median(), line_dash="dash",
              line_color="red", annotation_text=f"Median: ${ifood_df['MntTotal'].median():,.0f}")
fig.update_layout(showlegend=False, height=500)
fig.show()

print(f"Average customer spending (last 2 years): ${ifood_df['MntTotal'].mean():,.2f}")
print(f"Top 10% spenders (90th percentile, last 2 years): ${ifood_df['MntTotal'].quantile(0.90):,.2f}")


Average customer spending (last 2 years): $562.76
Top 10% spenders (90th percentile, last 2 years): $1,466.60


In [11]:
# Recency distribution
fig = px.histogram(ifood_df, x='Recency', nbins=40,
                   title='Distribution of Purchase Recency',
                   labels={'Recency': 'Days Since Last Purchase', 'count': 'Number of Customers'},
                   color_discrete_sequence=['mediumpurple'])
fig.add_vline(x=ifood_df['Recency'].median(), line_dash="dash", 
              line_color="red", annotation_text=f"Median: {ifood_df['Recency'].median():.0f} days")
fig.update_layout(showlegend=False, height=500)
fig.show()

## 4. Customer Demographics

Understanding who our customers are in terms of education, marital status, and family composition. This demographic information can help tailor marketing messages and product offerings to different customer segments.

In [12]:
# Marital status distribution - bar chart
marital_cols = [col for col in ifood_df.columns if col.startswith('marital_')]
marital_counts = ifood_df[marital_cols].sum().sort_values(ascending=False)
marital_counts.index = [col.replace('marital_', '') for col in marital_counts.index]

# Calculate percentages
marital_pct = (marital_counts / len(ifood_df) * 100).round(1)

fig = px.bar(
    x=marital_counts.index,
    y=marital_counts.values,
    title='Customer Distribution by Marital Status',
    labels={'x': 'Marital Status', 'y': 'Number of Customers'},
    color=marital_counts.values,
    color_continuous_scale='earth',
    text=[f"{count} ({pct}%)" for count, pct in zip(marital_counts.values, marital_pct.values)]
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()


In [13]:
print("\nMarital Status Distribution:")
for status, count in marital_counts.items():
    print(f"{status}: {count} ({count / len(ifood_df) * 100:.1f}%)")



Marital Status Distribution:
Married: 854 (38.7%)
Together: 568 (25.8%)
Single: 477 (21.6%)
Divorced: 230 (10.4%)
Widow: 76 (3.4%)


In [14]:
# Education level distribution
education_cols = [col for col in ifood_df.columns if col.startswith('education_')]
education_counts = ifood_df[education_cols].sum().sort_values(ascending=True)
education_counts.index = [col.replace('education_', '') for col in education_counts.index]

fig = px.bar(x=education_counts.index, y=education_counts.values,
              title='Customer Distribution by Education Level',
              labels={'x': 'Education Level', 'y': 'Number of Customers'},
              color_discrete_sequence=['steelblue'])
fig.update_layout(showlegend=False, height=400)
fig.show()

In [15]:
print("\nEducation Distribution:")
for edu, count in education_counts.items():
    print(f"{edu}: {count} ({count / len(ifood_df) * 100:.1f}%)")



Education Distribution:
Basic: 54 (2.4%)
2n Cycle: 198 (9.0%)
Master: 364 (16.5%)
PhD: 476 (21.6%)
Graduation: 1113 (50.5%)


#### Note:
The high share of customers in the *PhD* category is unexpected. The education labels also appear to follow a non‑U.S. system: **“2nd Cycle”** typically corresponds to a master’s level, and **“Graduation”** corresponds to a bachelor’s/undergraduate degree. We also cannot confirm whether this field represents each customer’s *highest* completed education, but that is a reasonable assumption given common data-collection practices.

A quick check confirms that **“2nd Cycle”** is a master’s-level designation under the Bologna Process: https://education.ec.europa.eu/education-levels/higher-education/inclusive-and-connected-higher-education/bologna-process. To make the results clearer for an international audience, we will:
- combine **2nd Cycle** and **Master** into a single **Master’s** group, and
- rename **Graduation** to **Bachelor’s**.


In [16]:
# Transform education categories
# Combine "2nd Cycle" and "Master" into "Master's"
ifood_df['education_Masters'] = ifood_df['education_2n Cycle'] + ifood_df['education_Master']

# Rename "Graduation" to "Bachelor's"
ifood_df['education_Bachelors'] = ifood_df['education_Graduation']

# Drop old education columns
ifood_df = ifood_df.drop(columns=['education_2n Cycle', 'education_Master', 'education_Graduation'])

# Update education distribution
education_cols = [col for col in ifood_df.columns if col.startswith('education_')]
education_counts = ifood_df[education_cols].sum().sort_values(ascending=True)
education_counts.index = [col.replace('education_', '') for col in education_counts.index]

In [17]:
#Visualize new categories
fig = px.bar(x=education_counts.index, y=education_counts.values,
              title='Customer Distribution by Education Level (Standardized)',
              labels={'x': 'Education Level', 'y': 'Number of Customers'},
              color_discrete_sequence=['steelblue'])
fig.update_layout(showlegend=False, height=400)
fig.show()

In [18]:
#Check percentages
print("\nStandardized Education Distribution:")
for edu, count in education_counts.items():
    print(f"{edu}: {count} ({count / len(ifood_df) * 100:.1f}%)")


Standardized Education Distribution:
Basic: 54 (2.4%)
PhD: 476 (21.6%)
Masters: 562 (25.5%)
Bachelors: 1113 (50.5%)


### Marital Status Distribution

Instead of a pie chart, we'll use a bar chart which makes it easier to compare values and see exact counts.

In [19]:
# Marital status distribution
marital_cols = [col for col in ifood_df.columns if col.startswith('marital_')]
marital_counts = ifood_df[marital_cols].sum().sort_values(ascending=False)
marital_counts.index = [col.replace('marital_', '') for col in marital_counts.index]

# Calculate percentages
marital_pct = (marital_counts / len(ifood_df) * 100).round(1)

fig = px.bar(x=marital_counts.index, y=marital_counts.values,
             title='Customer Distribution by Marital Status',
             labels={'x': 'Marital Status', 'y': 'Number of Customers'},
             color=marital_counts.values,
             color_continuous_scale='agsunset',
             text=[f"{count}<br>({pct}%)" for count, pct in zip(marital_counts.values, marital_pct.values)])
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [20]:
print("\nMarital Status Distribution:")
for status, count in marital_counts.items():
    print(f"{status}: {count} ({count/len(ifood_df)*100:.1f}%)")


Marital Status Distribution:
Married: 854 (38.7%)
Together: 568 (25.8%)
Single: 477 (21.6%)
Divorced: 230 (10.4%)
Widow: 76 (3.4%)


In [21]:
# Family composition - Kids and Teens
fig = make_subplots(rows=1, cols=2, subplot_titles=('Number of Kids at Home', 'Number of Teens at Home'))

kids_counts = ifood_df['Kidhome'].value_counts().sort_index()
teens_counts = ifood_df['Teenhome'].value_counts().sort_index()

fig.add_trace(go.Bar(x=kids_counts.index, y=kids_counts.values, 
                     marker_color='skyblue', name='Kids'),
              row=1, col=1)

fig.add_trace(go.Bar(x=teens_counts.index, y=teens_counts.values, 
                     marker_color='lightcoral', name='Teens'),
              row=1, col=2)

fig.update_xaxes(title_text="Number of Kids", row=1, col=1)
fig.update_xaxes(title_text="Number of Teens", row=1, col=2)
fig.update_yaxes(title_text="Number of Customers", row=1, col=1)
fig.update_layout(height=400, showlegend=False, title_text="Family Composition")
fig.show()

In [22]:

print(f"\nHouseholds with no children: {((ifood_df['Kidhome']==0) & (ifood_df['Teenhome']==0)).sum()} "
      f"({((ifood_df['Kidhome']==0) & (ifood_df['Teenhome']==0)).sum()/len(ifood_df)*100:.1f}%)")


Households with no children: 628 (28.5%)


## 5. Purchase Behavior Analysis

Analyzing spending patterns across product categories and purchase channels. This helps identify which products drive the most revenue and which channels customers prefer.

In [23]:
# Product category spending
product_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
product_labels = ['Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold Products']
product_totals = ifood_df[product_cols].sum()
product_totals.index = product_labels
product_totals = product_totals.sort_values(ascending=False)

fig = px.bar(x=product_totals.index, y=product_totals.values,
             title='Total Spending by Product Category',
             labels={'x': 'Product Category', 'y': 'Total Spending ($)'},
             color=product_totals.values,
             color_continuous_scale='Teal',
             text=product_totals.values)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [24]:

print("\nProduct Category Spending:")
total_spending = product_totals.sum()
for product, amount in product_totals.items():
    print(f"{product}: ${amount:,.0f} ({amount/total_spending*100:.1f}% of total)")


Product Category Spending:
Wines: $675,093 (50.5% of total)
Meat: $364,513 (27.2% of total)
Gold Products: $97,146 (7.3% of total)
Fish: $83,253 (6.2% of total)
Sweets: $59,818 (4.5% of total)
Fruits: $58,219 (4.4% of total)


In [25]:
# Average spending per customer by product category
product_means = ifood_df[product_cols].mean()
product_means.index = product_labels
product_means = product_means.sort_values(ascending=False)

fig = px.bar(x=product_means.index, y=product_means.values,
             title='Average Customer Spending by Product Category',
             labels={'x': 'Product Category', 'y': 'Average Spending per Customer ($)'},
             color=product_means.values,
             color_continuous_scale='Sunset',
             text=product_means.values)
fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [26]:
# Purchase channel analysis
purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
purchase_labels = ['Deals', 'Web', 'Catalog', 'Store']
purchase_totals = ifood_df[purchase_cols].sum()
purchase_totals.index = purchase_labels
purchase_totals = purchase_totals.sort_values(ascending=False)

fig = px.bar(x=purchase_totals.index, y=purchase_totals.values,
             title='Total Purchases by Channel',
             labels={'x': 'Purchase Channel', 'y': 'Number of Purchases'},
             color=purchase_totals.values,
             color_continuous_scale='turbo',
             text=purchase_totals.values)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [27]:

print("\nPurchase Channel Distribution:")
total_purchases = purchase_totals.sum()
for channel, count in purchase_totals.items():
    print(f"{channel}: {count:,} purchases ({count/total_purchases*100:.1f}%)")


Purchase Channel Distribution:
Store: 12,841 purchases (39.1%)
Web: 9,042 purchases (27.5%)
Catalog: 5,833 purchases (17.8%)
Deals: 5,112 purchases (15.6%)


In [28]:
# Web visits analysis
fig = px.histogram(ifood_df, x='NumWebVisitsMonth', nbins=20,
                   title='Distribution of Monthly Web Visits',
                   labels={'NumWebVisitsMonth': 'Number of Web Visits per Month', 'count': 'Number of Customers'},
                   color_discrete_sequence=['indianred'])
fig.update_layout(showlegend=False, height=500)
fig.show()

In [29]:

print(f"Average web visits per month: {ifood_df['NumWebVisitsMonth'].mean():.2f}")
print(f"Median web visits per month: {ifood_df['NumWebVisitsMonth'].median():.0f}")

Average web visits per month: 5.34
Median web visits per month: 6


## 6. Campaign Performance Analysis

Examining the effectiveness of different marketing campaigns. Understanding which campaigns performed best and what percentage of customers responded helps optimize future marketing efforts.

In [30]:
# Campaign acceptance rates
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
campaign_labels = ['Campaign 1', 'Campaign 2', 'Campaign 3', 'Campaign 4', 'Campaign 5', 'Latest Campaign']
campaign_acceptance = ifood_df[campaign_cols].sum()
campaign_acceptance.index = campaign_labels
campaign_acceptance = campaign_acceptance.sort_values(ascending=False)

fig = px.bar(x=campaign_acceptance.index, y=campaign_acceptance.values,
             title='Campaign Acceptance by Campaign',
             labels={'x': 'Campaign', 'y': 'Number of Acceptances'},
             color=campaign_acceptance.values,
             color_continuous_scale='Reds',
             text=campaign_acceptance.values)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [31]:

print("\nCampaign Acceptance Rates:")
for campaign, count in campaign_acceptance.items():
    rate = count / len(ifood_df) * 100
    print(f"{campaign}: {count} acceptances ({rate:.2f}%)")

print(f"\nOverall campaign acceptance (any campaign): {ifood_df['AcceptedCmpOverall'].sum()} customers")
print(f"Overall acceptance rate: {(ifood_df['AcceptedCmpOverall'] > 0).sum()/len(ifood_df)*100:.2f}%")


Campaign Acceptance Rates:
Latest Campaign: 333 acceptances (15.10%)
Campaign 4: 164 acceptances (7.44%)
Campaign 3: 163 acceptances (7.39%)
Campaign 5: 161 acceptances (7.30%)
Campaign 1: 142 acceptances (6.44%)
Campaign 2: 30 acceptances (1.36%)

Overall campaign acceptance (any campaign): 660 customers
Overall acceptance rate: 20.77%


In [32]:
# Distribution of number of campaigns accepted
campaign_counts = ifood_df['AcceptedCmpOverall'].value_counts().sort_index()

fig = px.bar(x=campaign_counts.index, y=campaign_counts.values,
             title='Distribution of Campaigns Accepted per Customer',
             labels={'x': 'Number of Campaigns Accepted', 'y': 'Number of Customers'},
             color=campaign_counts.values,
             color_continuous_scale='rdylgn',
             text=campaign_counts.values)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()

In [33]:

print(f"\nCustomers who accepted 0 campaigns: {campaign_counts[0]} ({campaign_counts[0]/len(ifood_df)*100:.1f}%)")
print(f"Customers who accepted 1+ campaigns: {(ifood_df['AcceptedCmpOverall'] > 0).sum()} ({(ifood_df['AcceptedCmpOverall'] > 0).sum()/len(ifood_df)*100:.1f}%)")


Customers who accepted 0 campaigns: 1747 (79.2%)
Customers who accepted 1+ campaigns: 458 (20.8%)


## 7. Correlation Analysis

Identifying relationships between key variables helps us understand which factors move together and might influence each other.

In [34]:
# Correlation heatmap for key features
key_features = ['Income', 'Age', 'Recency', 'MntTotal', 'NumWebVisitsMonth', 
                'AcceptedCmpOverall', 'Kidhome', 'Teenhome', 'Customer_Days']
corr_matrix = ifood_df[key_features].corr()

fig = px.imshow(corr_matrix, 
                text_auto='.2f',
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Correlation Heatmap of Key Features',
                labels=dict(color="Correlation"),
                aspect='auto')
fig.update_layout(height=600, width=700)
fig.show()

In [35]:

# Highlight strongest correlations
print("\nStrongest Correlations (absolute value):")
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for var1, var2, corr in corr_pairs[:5]:
    print(f"{var1} <-> {var2}: {corr:.3f}")


Strongest Correlations (absolute value):
Income <-> MntTotal: 0.823
Income <-> NumWebVisitsMonth: -0.648
MntTotal <-> Kidhome: -0.551
Income <-> Kidhome: -0.532
MntTotal <-> NumWebVisitsMonth: -0.502


## 8. Bivariate Relationships

Exploring relationships between pairs of variables that could drive business insights and inform segmentation strategies.

In [36]:
# Income vs Total Spending
fig = px.scatter(ifood_df, x='Income', y='MntTotal',
                 title='Income vs Total Spending',
                 labels={'Income': 'Annual Income ($)', 'MntTotal': 'Total Spending ($)'},
                 trendline='ols',
                 opacity=0.6,
                 color_discrete_sequence=['steelblue'])
fig.update_layout(height=500)
fig.show()

In [37]:

corr_value = ifood_df['Income'].corr(ifood_df['MntTotal'])
print(f"Correlation between Income and Total Spending: {corr_value:.3f}")
print(f"This indicates a {'strong' if abs(corr_value) > 0.7 else 'moderate' if abs(corr_value) > 0.4 else 'weak'} positive relationship.")

Correlation between Income and Total Spending: 0.823
This indicates a strong positive relationship.


In [38]:
# Age distribution by campaign acceptance
fig = px.box(ifood_df, x='AcceptedCmpOverall', y='Age',
             title='Age Distribution by Number of Campaigns Accepted',
             labels={'AcceptedCmpOverall': 'Number of Campaigns Accepted', 'Age': 'Age (years)'},
             color='AcceptedCmpOverall',
             color_discrete_sequence=px.colors.sequential.Viridis)
fig.update_layout(showlegend=False, height=500)
fig.show()

In [39]:

# Statistics
accepted_any = ifood_df[ifood_df['AcceptedCmpOverall'] > 0]['Age'].mean()
accepted_none = ifood_df[ifood_df['AcceptedCmpOverall'] == 0]['Age'].mean()
print(f"\nAverage age of customers who accepted campaigns: {accepted_any:.1f} years")
print(f"Average age of customers who didn't accept: {accepted_none:.1f} years")
print(f"Difference: {abs(accepted_any - accepted_none):.1f} years")


Average age of customers who accepted campaigns: 51.8 years
Average age of customers who didn't accept: 50.9 years
Difference: 0.9 years


In [40]:
# Income by campaign acceptance
fig = px.box(ifood_df, x='AcceptedCmpOverall', y='Income',
             title='Income Distribution by Number of Campaigns Accepted',
             labels={'AcceptedCmpOverall': 'Number of Campaigns Accepted', 'Income': 'Annual Income ($)'},
             color='AcceptedCmpOverall',
             color_discrete_sequence=px.colors.sequential.Teal)
fig.update_layout(showlegend=False, height=500)
fig.show()

In [41]:

# Statistics
income_accepted = ifood_df[ifood_df['AcceptedCmpOverall'] > 0]['Income'].mean()
income_not_accepted = ifood_df[ifood_df['AcceptedCmpOverall'] == 0]['Income'].mean()
print(f"\nAverage income of customers who accepted campaigns: ${income_accepted:,.2f}")
print(f"Average income of customers who didn't accept: ${income_not_accepted:,.2f}")
print(f"Difference: ${abs(income_accepted - income_not_accepted):,.2f} ({abs(income_accepted - income_not_accepted)/income_not_accepted*100:.1f}%)")


Average income of customers who accepted campaigns: $65,215.68
Average income of customers who didn't accept: $48,058.35
Difference: $17,157.33 (35.7%)


In [42]:
# Impact of children on spending - grouped by total children
ifood_df['TotalChildren'] = ifood_df['Kidhome'] + ifood_df['Teenhome']
children_spending = ifood_df.groupby('TotalChildren')['MntTotal'].mean().reset_index()

fig = px.bar(children_spending, x='TotalChildren', y='MntTotal',
             title='Average Spending by Total Number of Children',
             labels={'TotalChildren': 'Total Children (Kids + Teens)', 'MntTotal': 'Average Total Spending ($)'},
             color='MntTotal',
             color_continuous_scale='bluered',
             text='MntTotal')
fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False, height=500)
fig.show()


In [43]:
print("\nAverage spending by total number of children:")
for idx, row in children_spending.iterrows():
    print(f"{int(row['TotalChildren'])} children: ${row['MntTotal']:.2f}")


Average spending by total number of children:
0 children: $1041.21
1 children: $434.53
2 children: $221.57
3 children: $237.38


In [44]:
# Compare spending: no children vs with children
no_children_spending = children_spending[children_spending['TotalChildren'] == 0]['MntTotal'].values[0]
with_children_spending = children_spending[children_spending['TotalChildren'] > 0]['MntTotal'].mean()

print(f"\nAverage spending - No children: ${no_children_spending:.2f}")
print(f"Average spending - With children: ${with_children_spending:.2f}")
print(
    f"Difference: ${no_children_spending - with_children_spending:.2f} ({(no_children_spending - with_children_spending) / with_children_spending * 100:.1f}% higher)")



Average spending - No children: $1041.21
Average spending - With children: $297.83
Difference: $743.38 (249.6% higher)


In [45]:
# Web visits vs spending and campaign acceptance
fig = px.scatter(ifood_df, x='NumWebVisitsMonth', y='MntTotal',
                 color='AcceptedCmpOverall',
                 title='Web Visits vs Spending (colored by campaign acceptance)',
                 labels={'NumWebVisitsMonth': 'Monthly Web Visits', 
                         'MntTotal': 'Total Spending ($)',
                         'AcceptedCmpOverall': 'Campaigns Accepted'},
                 color_continuous_scale='Viridis',
                 opacity=0.6)
fig.update_layout(height=500)
fig.show()

In [46]:
#Investigate correlation further
corr_web_spending = ifood_df['NumWebVisitsMonth'].corr(ifood_df['MntTotal'])
print(f"Correlation between web visits and spending: {corr_web_spending:.3f}")
if corr_web_spending < 0:
    print("\n⚠ Interesting finding: Customers who visit the website more frequently tend to spend LESS!")
    print("   This could indicate that frequent visitors are browsing but not buying.")
    print("   Recommendation: Analyze user journeys to improve conversion rates.")

Correlation between web visits and spending: -0.502

⚠ Interesting finding: Customers who visit the website more frequently tend to spend LESS!
   This could indicate that frequent visitors are browsing but not buying.
   Recommendation: Analyze user journeys to improve conversion rates.


In [47]:
# Recency vs Spending
fig = px.scatter(ifood_df, x='Recency', y='MntTotal',
                 title='Purchase Recency vs Total Spending',
                 labels={'Recency': 'Days Since Last Purchase', 'MntTotal': 'Total Spending ($)'},
                 trendline='ols',
                 opacity=0.5,
                 color_discrete_sequence=['coral'])
fig.update_layout(height=500)
fig.show()

In [48]:
#Investigate correlation further
corr_recency = ifood_df['Recency'].corr(ifood_df['MntTotal'])
print(f"Correlation between recency and spending: {corr_recency:.3f}")

Correlation between recency and spending: 0.021


## 9. Outlier Detection

Identifying unusual data points using the IQR (Interquartile Range) method. Outliers may represent VIP customers, data entry errors, or special cases that require attention.

In [49]:
# Outlier detection using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check for outliers in key variables
outlier_summary = []
for col in ['Income', 'MntTotal', 'Age']:
    outliers, lower, upper = detect_outliers_iqr(ifood_df, col)
    outlier_summary.append({
        'Variable': col,
        'Outlier_Count': len(outliers),
        'Outlier_Pct': f"{len(outliers)/len(ifood_df)*100:.2f}%",
        'Lower_Bound': f"{lower:.2f}",
        'Upper_Bound': f"{upper:.2f}"
    })

outlier_df = pd.DataFrame(outlier_summary)
print("Outlier Detection Summary (IQR Method):")
print(outlier_df.to_string(index=False))

Outlier Detection Summary (IQR Method):
Variable  Outlier_Count Outlier_Pct Lower_Bound Upper_Bound
  Income              0       0.00%   -14431.50   117908.50
MntTotal              3       0.14%    -1306.00     2326.00
     Age              0       0.00%       16.00       88.00


In [50]:
# Visualize outliers
fig = make_subplots(rows=1, cols=3, subplot_titles=('Income', 'Total Spending', 'Age'))

fig.add_trace(go.Box(y=ifood_df['Income'], name='Income', marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Box(y=ifood_df['MntTotal'], name='Spending', marker_color='seagreen'), row=1, col=2)
fig.add_trace(go.Box(y=ifood_df['Age'], name='Age', marker_color='coral'), row=1, col=3)

fig.update_layout(height=500, showlegend=False, title_text="Boxplots for Outlier Detection")
fig.show()

## 10. Key Findings and Insights

### Data Quality ✓
- **No missing values** detected in the dataset, including Income
- Some very low income values exist (<$10k) but appear to be valid data points
- Outliers detected in Income, Spending, and Age variables may require treatment in modeling

### Customer Demographics
- Most customers are **graduates or have advanced degrees** (Master's/PhD)
- **Married customers** represent the largest marital status group (~39%)
- Significant portion of households have **no children**

### Spending Patterns 💰
- **Wines dominate** product spending (~50%), followed by **Meat Products** (~27%)
- **Strong positive correlation** between Income and Total Spending (r ≈ 0.82)
- Customers with children tend to **spend less** on average, perhaps due to the dominance of wines
- **Store purchases** are the most popular channel (~39%), followed by **Web purchases** (~28%)

### Campaign Performance 📊
- Overall campaign acceptance rate is **relatively low (~30%)**
- **Latest Campaign (Response)** had the highest acceptance (333 customers)
- Customers who accept campaigns tend to have **higher incomes**
- **Age shows some relationship** with campaign acceptance (older customers more responsive)

### Critical Insights ⚡
1. **Negative correlation** between web visits and spending suggests frequent browsers may not be converting to purchases
   - *Action*: Investigate user journey and improve website conversion
2. Recency shows weak correlation with total spending
3. Customer tenure (Customer_Days) shows minimal correlation with spending or campaign acceptance
4. Family composition significantly impacts spending behavior

### Recommendations for Next Steps 🎯
1. **Customer Segmentation**: Use clustering (K-means, hierarchical) to identify distinct customer groups
2. **Predictive Modeling**: Build classification models to predict campaign acceptance
3. **Feature Engineering**: Create RFM (Recency, Frequency, Monetary) scores for better segmentation
4. **Web Behavior Analysis**: Deep dive into why high web visitors spend less - A/B testing opportunities
5. **Product Affinity Analysis**: Analyze which products are purchased together (market basket analysis)
6. **Income Treatment**: Consider winsorization or log transformation for modeling due to outliers
7. **Targeted Campaigns**: Focus on high-income, no-children households who show strongest response

## Export Cleaned Data

Now that we've cleaned and transformed the data, let's save it to the processed folder for use in future analyses.

In [51]:
# Export cleaned dataframe to processed folder
output_path = "../data/processed/ifood_df_cleaned.csv"
ifood_df.to_csv(output_path, index=False)
print(f"✓ Cleaned data exported to: {output_path}")
print(f"  Shape: {ifood_df.shape}")
print(f"  Columns: {ifood_df.shape[1]}")
print(f"  Rows: {ifood_df.shape[0]}")

✓ Cleaned data exported to: ../data/processed/ifood_df_cleaned.csv
  Shape: (2205, 39)
  Columns: 39
  Rows: 2205
